# Discontinuity and 3D Trace Extraction

This notebook processes a point cloud using normal-based curvature, fuzzy c-means (FCM), DBSCAN, and PCA-based trace vectorization. It ends with saving 3D trace segments as `outputs/trace_lines_3d.shp` (with `.shx` and `.dbf` sidecars). All non-plane FCM clusters are defined as edge candidates in this published workflow.

## Setup and execution

- Run Jupyter from the repository root and run every code cell in order in one kernel. Close each Open3D window to continue. Open3D requires a desktop graphics session.
- Install the dependencies listed in `README.md`.
- Place the input LAS point cloud at `data/point_cloud.las`.
- Run the companion normal-generation notebook, then use the conversion snippet in `README.md` to create `data/point_cloud_normals.txt`. The last three columns must contain the normal components, with exactly one row per LAS point and the **same point order**; do not filter, reorder, or independently downsample either input.
- Intermediate exports and the curvature cache are written under `outputs/`. Coordinates are translated by the input XYZ minima before processing. The final export retains those translated coordinates because `global_offset=None`; pass the removed offset to the export function to restore world coordinates. The LAS-writing block in the final export function remains commented out.

The FCM, DBSCAN, and trace-extraction parameters are retained. Later spherical-window trace-length statistics are outside this notebook.


In [ ]:
import pandas as pd
import laspy
import numpy as np
import scipy
from scipy import spatial
import matplotlib
import matplotlib.pyplot as plt
import math
import open3d as o3d
from sklearn.cluster import DBSCAN
from collections import Counter
import numpy as np
from joblib import Parallel, delayed
from scipy.spatial import KDTree
from tqdm import tqdm
from sklearn.neighbors import KDTree
import skfuzzy as fuzz
from scipy.spatial import cKDTree

In [ ]:
def compute_curvature_vectorized(points, normals, k=30):
    """Mean unoriented angle between unit normals; process in batches with k=30."""
    unit_normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
    kdtree = cKDTree(points)
    curvatures = np.empty(len(points), dtype=np.float64)
    for start in range(0, len(points), 50000):
        end = min(start + 50000, len(points))
        _, indices = kdtree.query(points[start:end], k=k + 1, workers=-1)
        neighbor_indices = indices[:, 1:]
        dot_products = np.einsum('nj,nkj->nk', unit_normals[start:end], unit_normals[neighbor_indices])
        angle_diffs = np.arccos(np.clip(np.abs(dot_products), 0.0, 1.0))
        curvatures[start:end] = angle_diffs.mean(axis=1)
    return curvatures
def visualize_point_cloud(coords, colors, window_name="Point Cloud"):
    """Visualize the point cloud with Open3D."""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coords)
    # Open3D colors must be normalized to [0, 1].
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd], window_name=window_name)

In [ ]:
lasfile = r"data/point_cloud.las"
inFile = laspy.read(lasfile)
x0,y0,z0 = inFile.x,inFile.y,inFile.z
# r,g,b = inFile.red,inFile.green,inFile.blue  
# a = np.vstack((x0,y0,z0,r,g,b)).T
lasdata = np.vstack((x0,y0,z0)).T
lasdata = lasdata - np.min(lasdata, axis=0)

In [ ]:
nls = np.loadtxt(r"data/point_cloud_normals.txt", dtype=float, delimiter=' ')
nls = nls[:,-3:]
tree = spatial.cKDTree(lasdata) 
dist,nn_glob = tree.query(lasdata, k=30,workers=-1)

In [ ]:
len(nls)

In [ ]:
from pathlib import Path
Path("outputs").mkdir(parents=True, exist_ok=True)
curvatures = compute_curvature_vectorized(lasdata, nls)
np.save(r"outputs/point_cloud_curvature_axis_k30.npy", curvatures)

In [ ]:
curvatures = np.load(r"outputs/point_cloud_curvature_axis_k30.npy")
curvatures = np.array(curvatures)
curvatures = curvatures.reshape(-1,1)
len(curvatures)

In [ ]:
X_fcm = curvatures.reshape(1, -1) 

cntr, u, u0, d, jm, p, fpc = fuzz.cmeans(
    X_fcm, c=3, m=2.0, error=0.005, maxiter=1000, init=None, seed=20261014
)

In [ ]:
centers_fcm = cntr
print("=== FCM cluster centers ===")
print(centers_fcm)

In [ ]:
# Convert soft FCM memberships to hard labels using the maximum membership.
labels_fcm = np.argmax(u, axis=0)

# Normalize RGB values from [0, 255] to [0.0, 1.0].
plane_cluster_label_fcm = int(np.argmin(cntr.ravel()))
mask_fcm = labels_fcm == plane_cluster_label_fcm

# Define floating-point colors directly in [0.0, 1.0].
new_color1 = [43/255.0,  76/255.0,  89/255.0]
new_color2 = [219/255.0, 196/255.0, 155/255.0]

# Initialize the color array as float64.
colors_fcm = np.zeros_like(lasdata, dtype=np.float64)
colors_fcm[mask_fcm] = new_color1
colors_fcm[~mask_fcm] = new_color2

print("Displaying FCM clustering results...")
visualize_point_cloud(lasdata, colors_fcm, window_name="FCM Clustering Result")

In [ ]:
import laspy
import numpy as np
import os

# ================== Save the FCM results as an RGB LAS file ==================
output_fcm_las_path = r"outputs/fcm_clusters.las"

print(f"\nSaving the FCM point cloud and colors to: {output_fcm_las_path} ...")

# 1. Create a LAS header (point format 3 supports XYZ and RGB).
header = laspy.LasHeader(point_format=3, version="1.2")

# 2. Inherit scales and offsets from lasdata when available; otherwise compute them.
if 'lasdata' in locals() and hasattr(lasdata, 'header'):
    header.scales = lasdata.header.scales
    header.offsets = lasdata.header.offsets
elif 'lasdata' in globals() and hasattr(globals()['lasdata'], 'header'):
    header.scales = globals()['lasdata'].header.scales
    header.offsets = globals()['lasdata'].header.offsets
else:
    # Fallback: compute offsets from the current bounds and use 1 mm precision.
    print("No lasdata header was found; computing bounds and using millimeter precision...")
    header.offsets = np.min(lasdata, axis=0)
    header.scales = np.array([0.001, 0.001, 0.001])

# 3. Instantiate LasData.
las_out = laspy.LasData(header)

# 4. Write 3D coordinates from the lasdata point-position array.
las_out.x = lasdata[:, 0]
las_out.y = lasdata[:, 1]
las_out.z = lasdata[:, 2]

# 5. Write FCM cluster colors.
# colors_fcm is normalized to [0, 1]; map to uint16 [0, 65535] for LAS.
las_out.red = np.rint(np.clip(colors_fcm[:, 0], 0, 1) * 65535).astype(np.uint16)
las_out.green = np.rint(np.clip(colors_fcm[:, 1], 0, 1) * 65535).astype(np.uint16)
las_out.blue = np.rint(np.clip(colors_fcm[:, 2], 0, 1) * 65535).astype(np.uint16)

# 6. Write to disk.
las_out.write(output_fcm_las_path)

print(f"FCM point cloud saved successfully.")
print(f" -> Absolute file path: {os.path.abspath(output_fcm_las_path)}")
print(f" -> Total points: {len(lasdata)} points")
print(f" -> Lowest-indicator cluster / plane (color 1): {np.sum(mask_fcm)}; other clusters (color 2): {np.sum(~mask_fcm)}")

In [ ]:
raw_labels = np.argmax(u, axis=0)

center_values = cntr.flatten()
sort_indices = np.argsort(center_values)

sorted_centers = center_values[sort_indices]

label_mapping = {old_label: new_label for new_label, old_label in enumerate(sort_indices)}
labels_fcm_sorted = np.array([label_mapping[label] for label in raw_labels])

beautiful_palette = np.array([
    [43,  76,  89],   # Cluster 0 (planar): dark blue-gray
    [219, 196, 155],  # Cluster 1 (slight variation): sand
    [103, 144, 139],  # Cluster 2 (gradual transition): muted green
    [214, 130, 100],  # Cluster 3 (distinct edges): light orange
    # [153,  61,  70],  # Cluster 4 (sharp ridges/noise): reddish brown
    # [142, 202, 230]
    # [218, 215, 205]
])

# Original palette in [0, 255].
# Divide by 255.0 to map colors to [0.0, 1.0].
colors_fcm_multi = beautiful_palette[labels_fcm_sorted].astype(np.float64) / 255.0

print("\n=== Cluster statistics sorted by increasing curvature ===")
for i in range(len(sorted_centers)):
    cluster_count = np.sum(labels_fcm_sorted == i)
    print(f"Cluster {i} : {cluster_count:8d} points | Mean-curvature center: {sorted_centers[i]:.4f}")

print("\nDisplaying FCM clusters with sorted gradient colors...")
visualize_point_cloud(lasdata, colors_fcm_multi, window_name="FCM Sorted Gradient Result")

In [ ]:
import laspy
import numpy as np
from pathlib import Path

print("\n=== Saving each cluster as a separate LAS file ===")

# Output folder.
output_dir = Path(r"outputs/fcm_clusters")
output_dir.mkdir(parents=True, exist_ok=True)

# Number of clusters.
num_clusters = len(np.unique(labels_fcm_sorted))

for i in range(num_clusters):
    mask = (labels_fcm_sorted == i)
    cluster_points = lasdata[mask]

    point_count = len(cluster_points)
    if point_count == 0:
        continue

    header = laspy.LasHeader(point_format=3, version="1.2")
    header.offsets = np.min(cluster_points, axis=0)
    header.scales = np.array([0.001, 0.001, 0.001])

    new_las = laspy.LasData(header)
    new_las.x = cluster_points[:, 0]
    new_las.y = cluster_points[:, 1]
    new_las.z = cluster_points[:, 2]

    new_las.red = np.full(
        point_count, int(beautiful_palette[i][0] * 256), dtype=np.uint16
    )
    new_las.green = np.full(
        point_count, int(beautiful_palette[i][1] * 256), dtype=np.uint16
    )
    new_las.blue = np.full(
        point_count, int(beautiful_palette[i][2] * 256), dtype=np.uint16
    )

    output_filename = output_dir / f"cluster_{i}_curvature_{sorted_centers[i]:.4f}.las"
    new_las.write(output_filename)

    print(f"Saved: {output_filename} | Point count: {point_count}")


# Merge clusters 1, 2, 3, ... (exclude cluster 0).
other_mask = (labels_fcm_sorted != 0)
other_points = lasdata[other_mask]
other_labels = labels_fcm_sorted[other_mask]

if len(other_points) > 0:
    header = laspy.LasHeader(point_format=3, version="1.2")
    header.offsets = np.min(other_points, axis=0)
    header.scales = np.array([0.001, 0.001, 0.001])

    merged_las = laspy.LasData(header)
    merged_las.x = other_points[:, 0]
    merged_las.y = other_points[:, 1]
    merged_las.z = other_points[:, 2]

    # Assign the merged points a uniform color.
# All merged points use the color of cluster 1.
    merged_las.red = np.full(
        len(other_points), int(beautiful_palette[1][0] * 256), dtype=np.uint16
    )
    merged_las.green = np.full(
        len(other_points), int(beautiful_palette[1][1] * 256), dtype=np.uint16
    )
    merged_las.blue = np.full(
        len(other_points), int(beautiful_palette[1][2] * 256), dtype=np.uint16
    )
    merged_filename = output_dir / "merged_clusters_except_cluster_0.las"
    merged_las.write(merged_filename)

    print(f"Saved merged file: {merged_filename} | Point count: {len(other_points)}")
else:
    print("No points outside cluster 0; no merged file was created.")

print("=== All cluster point clouds saved. ===")

In [ ]:
# All sorted non-plane FCM clusters are edge candidates in this published workflow.
intersection_edge_mask = labels_fcm_sorted > 0
colors_intersection = colors_fcm_multi.copy()

edge_mask = intersection_edge_mask.copy()

# Apply the mask to extract 3D coordinates and the assigned colors.
edge_coords = lasdata[edge_mask]
edge_colors = colors_intersection[edge_mask]

In [ ]:

# Recompute the removed offset from the minima of the original input coordinates.
offset = np.array([inFile.x.min(), inFile.y.min(), inFile.z.min()])
# 1. Restore the original world coordinates.
edge_coords_real_world = edge_coords + offset

print(f"Preparing export... extracted {len(edge_coords_real_world)} edge points.")

# 2. Create a LAS header.
header = laspy.LasHeader(point_format=3, version="1.3")
header.offsets = inFile.header.offsets
header.scales = inFile.header.scales

# 3. Populate the LAS object with the restored coordinates.
out_las = laspy.LasData(header)
out_las.x = edge_coords_real_world[:, 0]
out_las.y = edge_coords_real_world[:, 1]
out_las.z = edge_coords_real_world[:, 2]

# 4. Write color data.
# colors_fcm_multi is in [0.0, 1.0].
# LAS RGB values are uint16 [0, 65535], so multiply by 65535.
out_las.red = (edge_colors[:, 0] * 65535).astype(np.uint16)
out_las.green = (edge_colors[:, 1] * 65535).astype(np.uint16)
out_las.blue = (edge_colors[:, 2] * 65535).astype(np.uint16)

# 5. Output file.
save_path = r"outputs/edge_candidates_world.las"
out_las.write(save_path)

print(f"Export complete. Saved to: {save_path}")

In [ ]:
import numpy as np
import open3d as o3d
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# lasdata and labels_fcm_sorted are the preceding sorted FCM results.
# Cluster 0 contains plane candidates; higher labels contain edge candidates.

print("1. Merging edge candidates...")
# Treat all points with sorted FCM labels greater than 0 as edge candidates.
mask_edges = intersection_edge_mask.copy()
edge_points = lasdata[mask_edges]

print(f"Merged edge-candidate points: {len(edge_points)} (fraction of input: {len(edge_points)/len(lasdata)*100:.1f}%)")

# ================== DBSCAN parameters ==================
# eps: neighborhood radius, in the same units as the coordinates.
# min_samples: minimum neighborhood size for a DBSCAN core point.
eps_value = 0.12       # Search radius in coordinate units.
min_samples_value = 4 # Minimum number of neighbors for a core point.

print(f"2. Running DBSCAN (eps={eps_value}, min_samples={min_samples_value})...")
dbscan_model = DBSCAN(eps=eps_value, min_samples=min_samples_value, n_jobs=-1)
db_labels = dbscan_model.fit_predict(edge_points)

# ================== Parse DBSCAN results ==================
# Label -1 denotes noise.
mask_noise = db_labels == -1
mask_core = ~mask_noise

noise_points = edge_points[mask_noise]
core_points = edge_points[mask_core]
valid_labels = db_labels[mask_core]

# Count independent trace/skeleton clusters after excluding noise (-1).
if len(valid_labels) > 0:
    num_clusters = valid_labels.max() + 1
else:
    num_clusters = 0

print(f"DBSCAN complete:")
print(f" -> Removed noise points: {len(noise_points)}")
print(f" -> Retained skeleton points: {len(core_points)}")
print(f" -> Independent discontinuity-trace/skeleton segments: {num_clusters} segments")

# ================== 3. Assign a color to each skeleton cluster ==================
# Assign random colors to retained clusters and dark gray to noise.
colors_dbscan = np.zeros_like(edge_points, dtype=np.float64)

# 1. Show noise in dark gray.
colors_dbscan[mask_noise] = [0.3, 0.3, 0.3]

# 2. Generate distinct random colors for valid clusters.
if num_clusters > 0:
    # Generate a color palette in HSV space.
    cmap = plt.get_cmap("hsv")
    # Sample uniformly and shuffle to reduce similarity between adjacent clusters.
    cluster_colors = cmap(np.linspace(0, 1, num_clusters))[:, :3]
    np.random.shuffle(cluster_colors)
    
    # Map colors to their points.
    colors_dbscan[mask_core] = cluster_colors[valid_labels]

# ================== Open3D visualization ==================
print("Displaying DBSCAN results (gray: noise; colors: connected skeleton clusters)...")
pcd_dbscan = o3d.geometry.PointCloud()
pcd_dbscan.points = o3d.utility.Vector3dVector(edge_points)
pcd_dbscan.colors = o3d.utility.Vector3dVector(colors_dbscan)

# Optional: display the original plane candidates as context.
# plane_points = lasdata[~mask_edges]
# plane_colors = np.tile([0.8, 0.8, 0.8], (len(plane_points), 1)) # Light-gray planes
# pcd_plane = o3d.geometry.PointCloud()
# pcd_plane.points = o3d.utility.Vector3dVector(plane_points)
# pcd_plane.colors = o3d.utility.Vector3dVector(plane_colors)
# o3d.visualization.draw_geometries([pcd_dbscan, pcd_plane], window_name="DBSCAN Trace Clusters")

o3d.visualization.draw_geometries([pcd_dbscan], window_name="DBSCAN Trace Clusters (Edges Only)")

In [ ]:
# 1. Extract coordinates retained after DBSCAN noise removal.
clean_edge_coords = edge_points[mask_core]

# 2. Extract the corresponding colors.
# The edge-candidate mask is labels_fcm_sorted > 0.
# Retrieve the original normalized FCM colors of all edge candidates.
original_edge_colors = colors_intersection[mask_edges]

# Apply the same DBSCAN mask to remove colors associated with noise.
clean_edge_colors = original_edge_colors[mask_core]

In [ ]:

# 3. Create a LAS header (format 3 supports RGB).
header = laspy.LasHeader(point_format=3, version="1.2")

# Set offsets and scales to preserve coordinate precision.
header.offsets = np.min(clean_edge_coords, axis=0)
header.scales = np.array([0.001, 0.001, 0.001])

# 4. Populate the LAS object with coordinates after noise removal.
out_las = laspy.LasData(header)
out_las.x = clean_edge_coords[:, 0]
out_las.y = clean_edge_coords[:, 1]
out_las.z = clean_edge_coords[:, 2]

# 5. Apply the existing RGB conversion from 8-bit to 16-bit values.
out_las.red = (clean_edge_colors[:, 0] * 256).astype(np.uint16)
out_las.green = (clean_edge_colors[:, 1] * 256).astype(np.uint16)
out_las.blue = (clean_edge_colors[:, 2] * 256).astype(np.uint16)

# 6. Save to disk.
output_path = r"outputs/edge_points_denoised.las"
out_las.write(output_path)

print(f"Saved {len(clean_edge_coords)} edge points after noise removal to: {output_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import colorsys
import numpy as np

def generate_distinct_colors(num_colors):
    """Generate distinct RGB colors using golden-ratio hue steps, alternating saturation/value, and random ordering."""
    if num_colors <= 0:
        return np.empty((0, 3), dtype=np.float64)
        
    colors = np.zeros((num_colors, 3), dtype=np.float64)
    
    # Golden-ratio conjugate for hue stepping.
    golden_ratio_conjugate = 0.618033988749895
    h = np.random.rand() # Random initial hue.
    
    # Preset high-contrast saturation (S) and value (V) profiles.
    # Alternate profiles to distinguish colors with similar hues.
    sv_profiles = [
        (0.90, 0.95),  # Vivid
        (0.65, 0.95),  # Pastel
        (0.95, 0.70),  # Deep
        (0.45, 0.95),  # Light
        (0.80, 0.85),  # Standard
    ]
    
    for i in range(num_colors):
        # Golden-ratio hue step.
        h = (h + golden_ratio_conjugate) % 1.0
        
        # Cycle through the preset S/V profiles.
        s, v = sv_profiles[i % len(sv_profiles)]
        
        # Convert HSV to RGB.
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        colors[i] = [r, g, b]
        
    # Shuffle colors to randomize their assignment.
    # This reduces sequential hue/profile assignments to neighboring point-cloud clusters.
    np.random.shuffle(colors)
    
    return colors

# Initialize an all-False global mask with the same length as the input point cloud.
global_valid_edge_mask = np.zeros(len(lasdata), dtype=bool)

# Mark the points retained by the preceding DBSCAN step in the global mask.
global_valid_edge_mask[mask_edges] = mask_core 

# Use only FCM cluster 0 as plane candidates; non-plane FCM clusters remain edge candidates.
mask_remaining = labels_fcm_sorted == 0
plane_points = lasdata[mask_remaining]

print(f"Original FCM plane-candidate points: {len(plane_points)} (fraction of input: {len(plane_points)/len(lasdata)*100:.1f}%)")


In [ ]:
eps_plane = 0.07
min_samples_plane = 4

print(f"2. Running DBSCAN on plane candidates (eps={eps_plane}, min_samples={min_samples_plane})...")
dbscan_plane = DBSCAN(eps=eps_plane, min_samples=min_samples_plane, n_jobs=-1)
plane_labels = dbscan_plane.fit_predict(plane_points)


# ================== 3. Parse plane clusters ==================
mask_plane_noise = plane_labels == -1
mask_plane_core = ~mask_plane_noise

valid_plane_labels = plane_labels[mask_plane_core]

if len(valid_plane_labels) > 0:
    num_plane_clusters = valid_plane_labels.max() + 1
else:
    num_plane_clusters = 0

print(f"Plane DBSCAN complete:")
print(f" -> Noise points/fragments: {np.sum(mask_plane_noise)}")
print(f" -> Independent discontinuity surfaces/planes: {num_plane_clusters} planes")


# ================== 4. Assign colors to independent planes ==================
# Initialize the color array.
colors_plane = np.zeros_like(plane_points, dtype=np.float64)

# 1. Show noise fragments in dark gray.
colors_plane[mask_plane_noise] = [0.4, 0.4, 0.4]

# 2. Generate distinct high-contrast colors for the plane clusters.
if num_plane_clusters > 0:
    # Generate the required number of colors with the golden-ratio function.
    cluster_colors_plane = generate_distinct_colors(num_plane_clusters)

    # Map generated colors directly to plane labels.
    colors_plane[mask_plane_core] = cluster_colors_plane[valid_plane_labels]

print("\nDisplaying the separated planes...")
pcd_plane = o3d.geometry.PointCloud()
pcd_plane.points = o3d.utility.Vector3dVector(plane_points)
pcd_plane.colors = o3d.utility.Vector3dVector(colors_plane)
o3d.visualization.draw_geometries([pcd_plane], window_name="Separated Planes (Distinct Colors)")

In [ ]:
import laspy
import os
import numpy as np

# ================== 5. Save an RGB LAS file excluding gray noise ==================
output_las_path = r"outputs/plane_clusters.las"
print(f"\n5. Saving non-gray plane points and colors to: {output_las_path} ...")

# Treat points with little difference among RGB channels as gray noise.
GRAY_TOLERANCE = 0.01
color_span = np.ptp(colors_plane[:, :3], axis=1)  # Per-point max(R, G, B) - min(R, G, B).
keep_mask = np.isfinite(plane_points).all(axis=1) & np.isfinite(colors_plane[:, :3]).all(axis=1)
keep_mask &= color_span > GRAY_TOLERANCE

export_points = plane_points[keep_mask]
export_colors = colors_plane[keep_mask, :3]

if len(export_points) == 0:
    raise ValueError("No non-gray points remain for export; check colors_plane or the GRAY_TOLERANCE setting.")

print(f" -> Removed gray noise points: {(~keep_mask).sum()} points")
print(f" -> Non-gray points to save: {len(export_points)} points")

# 1. Create a LAS header.
header = laspy.LasHeader(point_format=3, version="1.2")

# 2. Retain the original LAS precision if available; otherwise use millimeter precision.
if 'lasdata' in locals() and hasattr(lasdata, 'header'):
    header.scales = lasdata.header.scales
    header.offsets = lasdata.header.offsets
elif 'lasdata' in globals() and hasattr(globals()['lasdata'], 'header'):
    header.scales = globals()['lasdata'].header.scales
    header.offsets = globals()['lasdata'].header.offsets
else:
    header.offsets = np.min(export_points, axis=0)
    header.scales = np.array([0.001, 0.001, 0.001])

# 3. Write coordinates and RGB values.
las_out = laspy.LasData(header)
las_out.x = export_points[:, 0]
las_out.y = export_points[:, 1]
las_out.z = export_points[:, 2]

las_out.red = (np.clip(export_colors[:, 0], 0, 1) * 65535).astype(np.uint16)
las_out.green = (np.clip(export_colors[:, 1], 0, 1) * 65535).astype(np.uint16)
las_out.blue = (np.clip(export_colors[:, 2], 0, 1) * 65535).astype(np.uint16)

os.makedirs(os.path.dirname(output_las_path), exist_ok=True)
las_out.write(output_las_path)

print("Saved successfully.")
print(f" -> File path: {os.path.abspath(output_las_path)}")
print(f" -> Total points: {len(export_points)} points")

In [ ]:

# ================== Apply to the preceding plane candidates ==================

# 1. Extract normals corresponding to these plane points.
# mask_remaining selects plane candidates; nls contains global point normals.
plane_normals = nls[mask_remaining]

In [ ]:
import pandas as pd
import math

# ================== 3. PCA normals and dip direction/dip ==================
print("\n3. Fitting PCA normals and calculating geological orientation (dip direction/dip)...")

# Store orientation results.
orientation_results = []

for cluster_id in range(num_plane_clusters):
    # Extract the 3D coordinates of the current plane.
    idx = np.where(plane_labels == cluster_id)[0]
    pts = plane_points[idx]
    
    # Apply the existing minimum of 20 points for the plane fit.
    if len(pts) < 20:
        continue 
        
    # --- Step 1: Fit a plane normal with PCA ---
    # Calculate the plane-patch centroid.
    centroid = np.mean(pts, axis=0)
    # Center the coordinates.
    centered_pts = pts - centroid
    # Compute the 3 x 3 covariance matrix.
    cov_matrix = np.dot(centered_pts.T, centered_pts) / (len(pts) - 1)
    
    # Eigenvalue decomposition.
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    # eigh sorts eigenvalues in ascending order; the first eigenvector is the plane normal.
    normal = eigenvectors[:, 0] 
    
    # --- Step 2: Orient normals upward (Z > 0) ---
    if normal[2] < 0:
        normal = -normal
        
    nx, ny, nz = normal

    # --- Step 3: Calculate dip angle ---
    # The angle to the Z axis is arccos(nz / |normal|); the normal is a unit vector.
    dip_angle_rad = np.arccos(nz)
    dip_angle = np.degrees(dip_angle_rad)

    # --- Step 4: Calculate dip direction ---
    # Coordinate convention: X east, Y north, Z up.
    # The horizontal dip-direction vector is (-nx, -ny).
    # math.atan2(x, y) gives azimuth clockwise from north (Y).
    dip_direction_rad = math.atan2(-nx, -ny)
    dip_direction = np.degrees(dip_direction_rad)
    
    # Map the azimuth to [0, 360) degrees.
    dip_direction = (dip_direction + 360) % 360

    # Record the current plane results.
    orientation_results.append({
        "Cluster ID": cluster_id,
        "Point Count": len(pts),
        "Centroid_X": round(centroid[0], 3),
        "Centroid_Y": round(centroid[1], 3),
        "Centroid_Z": round(centroid[2], 3),
        "Normal_Nx": round(nx, 4),
        "Normal_Ny": round(ny, 4),
        "Normal_Nz": round(nz, 4),
        "Dip Direction / degrees": round(dip_direction, 2),
        "Dip Angle / degrees": round(dip_angle, 2)
    })

# ================== 4. Export results to Excel ==================
if orientation_results:
    # Convert to a pandas DataFrame.
    df_orientations = pd.DataFrame(orientation_results)
    
    # Output filename.
    output_excel_path = "outputs/plane_orientations.xlsx"
    
    # Export.
    df_orientations.to_excel(output_excel_path, index=False)
    print(f" -> Calculated orientations for {len(orientation_results)} discontinuity surfaces.")
    print(f" -> Results exported to Excel: {output_excel_path}\n")
else:
    print(" -> No valid discontinuity surfaces were found for orientation calculations.")

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import open3d as o3d
from collections import Counter
from tqdm import tqdm

print("\n1. Preparing plane data for ridge/valley classification...")

# Exclude plane points labeled as noise (-1).
valid_plane_mask = plane_labels != -1
valid_plane_points = plane_points[valid_plane_mask]
valid_plane_normals = plane_normals[valid_plane_mask] # plane_normals was defined above.
valid_plane_labels = plane_labels[valid_plane_mask]

# Build a KDTree of plane points to associate edges with two neighboring planes.
plane_tree = cKDTree(valid_plane_points)

# Find the k nearest plane points around each edge; k=60 is used to span gaps.
k_neighbors = 60 
print(f"2. Querying neighboring planes for edge points (k={k_neighbors})...")
distances, indices = plane_tree.query(clean_edge_coords, k=k_neighbors, workers=-1)

# Initialize edge types: 0 unknown/boundary, 1 ridge (convex), 2 valley (concave).
edge_types = np.zeros(len(clean_edge_coords), dtype=int)

print("3. Performing geometric-topological ridge/valley classification...")
for i in tqdm(range(len(clean_edge_coords)), desc="Classifying Edges"):
    neighbor_idxs = indices[i]
    neighbor_labels = valid_plane_labels[neighbor_idxs]
    
    # Find the two most frequent neighboring plane labels.
    label_counts = Counter(neighbor_labels)
    most_common = label_counts.most_common(2)
    
    # Classification requires two planes; a single plane may indicate an outer boundary.
    if len(most_common) < 2:
        continue
        
    label_A = most_common[0][0]
    label_B = most_common[1][0]
    
    # Extract local points and normals belonging to planes A and B.
    mask_A = neighbor_labels == label_A
    mask_B = neighbor_labels == label_B
    
    pts_A = valid_plane_points[neighbor_idxs][mask_A]
    nrm_A = valid_plane_normals[neighbor_idxs][mask_A]
    
    pts_B = valid_plane_points[neighbor_idxs][mask_B]
    nrm_B = valid_plane_normals[neighbor_idxs][mask_B]
    
    # Calculate local centroids and mean normals.
    c_A = np.mean(pts_A, axis=0)
    n_A = np.mean(nrm_A, axis=0)
    n_A /= np.linalg.norm(n_A)  # Normalize.
    
    c_B = np.mean(pts_B, axis=0)
    n_B = np.mean(nrm_B, axis=0)
    n_B /= np.linalg.norm(n_B)  # Normalize.
    
    # Vector from the centroid of plane A to that of plane B.
    v_ab = c_B - c_A
    
    # Topological classification using a projection score.
    score = np.dot(n_A, v_ab) - np.dot(n_B, v_ab)
    
    if score < 0:
        edge_types[i] = 1  # Ridge / convex
    else:
        edge_types[i] = 2  # Valley / concave


# ================== 4. Prepare classification visualization ==================
print("\n4. Preparing the visualization...")

# Colors.
color_ridge = np.array([1.0, 0.2, 0.2])   # Red: ridge (convex).
color_valley = np.array([0.2, 0.5, 1.0])  # Blue: valley (concave).
color_unknown = np.array([0.5, 0.5, 0.5]) # Gray: single-plane boundary.

edge_colors = np.zeros_like(clean_edge_coords, dtype=np.float64)
edge_colors[edge_types == 0] = color_unknown
edge_colors[edge_types == 1] = color_ridge
edge_colors[edge_types == 2] = color_valley

# Counts.
num_ridges = np.sum(edge_types == 1)
num_valleys = np.sum(edge_types == 2)
print(f" -> Ridge points: {num_ridges}")
print(f" -> Valley points: {num_valleys}")

# Show edge points together with darkened plane points for context.
context_plane_colors = colors_plane * 0.3 # Darken planes to highlight the lines.
pcd_planes = o3d.geometry.PointCloud()
pcd_planes.points = o3d.utility.Vector3dVector(plane_points)
pcd_planes.colors = o3d.utility.Vector3dVector(context_plane_colors)

pcd_edges = o3d.geometry.PointCloud()
pcd_edges.points = o3d.utility.Vector3dVector(clean_edge_coords)
pcd_edges.colors = o3d.utility.Vector3dVector(edge_colors)

# Display darkened planes and colored ridge/valley points in Open3D.
o3d.visualization.draw_geometries([pcd_planes, pcd_edges], window_name="Ridge (Red) vs Valley (Blue) Classification")

In [ ]:
import numpy as np
import open3d as o3d
from collections import defaultdict

def vectorize_traces_by_topology(edge_coords, edge_types, edge_topology, plane_normals, plane_labels, gap_threshold=0.5):
    """Vectorize traces using plane-pair topology and intersection directions. gap_threshold is the maximum gap (coordinate units) between projected points within one extracted segment."""
    print("Extracting vector traces using plane-pair topology...")
    
    # Group edge points by the pair of neighboring planes (A, B).
    # frozenset groups (A, B) and (B, A) together.
    topology_groups = defaultdict(list)
    for idx, (type_val, (label_A, label_B)) in enumerate(zip(edge_types, edge_topology)):
        if type_val == 0: continue # Skip unknown or single-plane boundary points.
        pair = frozenset([label_A, label_B])
        topology_groups[pair].append(idx)
        
    line_segments = []
    line_types = [] # Store the ridge/valley type of each extracted line.
    
    # Build a lookup from unique plane labels to mean normals.
    # plane_normals contains point normals grouped by their plane labels.
    unique_labels = np.unique(plane_labels)
    normal_dict = {}
    for lbl in unique_labels:
        if lbl == -1: continue
        # Average all point normals belonging to the plane.
        avg_n = np.mean(plane_normals[plane_labels == lbl], axis=0)
        normal_dict[lbl] = avg_n / np.linalg.norm(avg_n)

    for pair, point_indices in topology_groups.items():
        if len(pair) < 2 or len(point_indices) < 5:
            continue # Insufficient points to form a valid segment.
            
        lbl_A, lbl_B = tuple(pair)
        pts = edge_coords[point_indices]
        e_type = edge_types[point_indices[0]] # Use the edge type of the first point in the group.
        
        # 1. Calculate the intersection direction using a cross product.
        n_A = normal_dict[lbl_A]
        n_B = normal_dict[lbl_B]
        direction = np.cross(n_A, n_B)
        dir_norm = np.linalg.norm(direction)
        
        if dir_norm < 1e-5:
            continue # The planes are nearly parallel; skip their intersection.
        direction /= dir_norm
        
        # 2. Use the edge-point centroid as the reference point of the 3D line.
        centroid = np.mean(pts, axis=0)
        
        # 3. Project edge points onto the line to obtain scalar positions t.
        # Projection: t = (P - C) dot D.
        t_values = np.dot(pts - centroid, direction)
        
        # Sort projected positions for gap detection and segment extraction.
        sorted_indices = np.argsort(t_values)
        t_sorted = t_values[sorted_indices]
        
        # 4. Detect gaps and generate multiple line segments.
        segment_start_idx = 0
        for i in range(1, len(t_sorted)):
            # A projected spacing above the threshold starts a new segment.
            if t_sorted[i] - t_sorted[i-1] > gap_threshold:
                # Extract the preceding segment if it contains enough points.
                if i - segment_start_idx >= 3:
                    t_start = t_sorted[segment_start_idx]
                    t_end = t_sorted[i-1]
                    start_pt = centroid + t_start * direction
                    end_pt = centroid + t_end * direction
                    line_segments.append([start_pt, end_pt])
                    line_types.append(e_type)
                segment_start_idx = i # Start a new segment.
                
        # Extract the final segment.
        if len(t_sorted) - segment_start_idx >= 3:
            t_start = t_sorted[segment_start_idx]
            t_end = t_sorted[-1]
            start_pt = centroid + t_start * direction
            end_pt = centroid + t_end * direction
            line_segments.append([start_pt, end_pt])
            line_types.append(e_type)

    print(f"Extracted vector segments: {len(line_segments)} segments")
    return line_segments, line_types

# ================== Vector-line visualization ==================
# After execution, line_segments and line_types contain the extracted geometry and types.
# line_segments has the structure [[p1, p2], [p3, p4], ...].

In [ ]:
import numpy as np
import open3d as o3d
from scipy.spatial import cKDTree
from collections import defaultdict, Counter
from tqdm import tqdm

def extract_traces_pca_topology(edge_points, plane_points, plane_labels, k_neighbors=60, gap_threshold=0.5):
    """Extract vector traces by PCA plane normals, plane-pair topology, and projection along intersection directions."""
    print("1. Computing a PCA normal for each independent discontinuity surface...")
    unique_labels = np.unique(plane_labels)
    pca_normals = {}
    
    for lbl in unique_labels:
        if lbl == -1: 
            continue # Skip noise labels.
            
        pts = plane_points[plane_labels == lbl]
        if len(pts) < 3:
            continue
            
        # PCA calculation.
        centroid = np.mean(pts, axis=0)
        centered_pts = pts - centroid
        # Compute the 3 x 3 covariance matrix.
        cov_matrix = np.cov(centered_pts, rowvar=False)
        # Eigenvalue decomposition.
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        # The eigenvector of the smallest eigenvalue is the best-fit plane normal.
        normal = eigenvectors[:, 0]
        normal = normal / np.linalg.norm(normal)
        pca_normals[lbl] = normal

    print("2. Determining edge-to-plane-pair topology using a KDTree...")
    plane_tree = cKDTree(plane_points)
    _, indices = plane_tree.query(edge_points, k=k_neighbors, workers=-1)
    
    # Store edge-point indices for each plane pair (A, B).
    topology_groups = defaultdict(list)
    
    for i in tqdm(range(len(edge_points)), desc="Mapping Edges to Planes"):
        neighbor_labels = plane_labels[indices[i]]
        # Exclude noise labels (-1).
        valid_neighbors = neighbor_labels[neighbor_labels != -1]
        
        if len(valid_neighbors) < 2:
            continue
            
        label_counts = Counter(valid_neighbors)
        most_common = label_counts.most_common(2)
        
        if len(most_common) == 2:
            lbl_A, lbl_B = most_common[0][0], most_common[1][0]
            # frozenset groups (A, B) and (B, A) together.
            pair = frozenset([lbl_A, lbl_B])
            topology_groups[pair].append(i)

    print(f" -> Found {len(topology_groups)} plane-pair topology groups.")

    print("3. Computing intersection directions and projecting edge points...")
    line_segments = []
    
    for pair, point_indices in topology_groups.items():
        if len(pair) < 2 or len(point_indices) < 5:
            continue # Insufficient points to form a reliable trace.
            
        lbl_A, lbl_B = tuple(pair)
        
        # Ensure PCA normals are available for both planes.
        if lbl_A not in pca_normals or lbl_B not in pca_normals:
            continue
            
        n_A = pca_normals[lbl_A]
        n_B = pca_normals[lbl_B]
        
        # Step 1: Cross the two PCA normals to obtain the intersection direction.
        direction = np.cross(n_A, n_B)
        dir_norm = np.linalg.norm(direction)
        if dir_norm < 1e-5:
            continue # The planes are nearly parallel; skip their intersection.
        direction /= dir_norm
        
        # Extract all edge points associated with this plane pair.
        group_edge_points = edge_points[point_indices]
        
        # Step 2: Use the edge-point centroid to locate the 3D projection line.
        group_centroid = np.mean(group_edge_points, axis=0)
        
        # Step 3: Project edge points onto the line to obtain scalar positions t.
        # Formula: t = (P - C) dot D.
        t_values = np.dot(group_edge_points - group_centroid, direction)
        
        # Sort by projected position.
        t_sorted = np.sort(t_values)
        
        # Step 4: Detect gaps and generate segments using gap_threshold.
        segment_start_idx = 0
        for i in range(1, len(t_sorted)):
            if t_sorted[i] - t_sorted[i-1] > gap_threshold:
                # At a gap, extract the preceding segment.
                if i - segment_start_idx >= 3: # Exclude very short isolated segments.
                    t_start = t_sorted[segment_start_idx]
                    t_end = t_sorted[i-1]
                    p_start = group_centroid + t_start * direction
                    p_end = group_centroid + t_end * direction
                    line_segments.append([p_start, p_end])
                segment_start_idx = i # Start the next segment.
                
        # Extract the final segment.
        if len(t_sorted) - segment_start_idx >= 3:
            t_start = t_sorted[segment_start_idx]
            t_end = t_sorted[-1]
            p_start = group_centroid + t_start * direction
            p_end = group_centroid + t_end * direction
            line_segments.append([p_start, p_end])

    print(f"PCA-based vector segments extracted: {len(line_segments)} segments")
    return line_segments


# ================== Execute and visualize ==================
# Use cleaned edge points and the plane points with their supplied labels.
# gap_threshold = 0.5 splits traces when adjacent projected points are more than 0.5 coordinate units apart.
final_lines = extract_traces_pca_topology(
    edge_points=clean_edge_coords, 
    plane_points=plane_points, 
    plane_labels=plane_labels,
    k_neighbors=60, 
    gap_threshold=0.5  
)

In [ ]:

# Build an Open3D line set for visualization.
line_points = []
line_indices = []

current_idx = 0
for segment in final_lines:
    line_points.extend([segment[0], segment[1]])
    line_indices.append([current_idx, current_idx + 1])
    current_idx += 2

line_set = o3d.geometry.LineSet()
line_set.points = o3d.utility.Vector3dVector(line_points)
line_set.lines = o3d.utility.Vector2iVector(line_indices)
# Color all traces red.
line_set.paint_uniform_color([1.0, 0.0, 0.0])

# Visualization: red traces, with optional darkened planes as context.
# context_plane_colors = colors_plane * 0.4
# pcd_planes = o3d.geometry.PointCloud()
# pcd_planes.points = o3d.utility.Vector3dVector(plane_points)
# pcd_planes.colors = o3d.utility.Vector3dVector(context_plane_colors)

o3d.visualization.draw_geometries([line_set], window_name="PCA-driven Trace Vectors")

In [ ]:
import laspy
import shapefile
import numpy as np
import os

def save_las_and_shp_with_offset(plane_points, line_segments,
                                 las_filename="output_cloud.las",
                                 shp_filename="output_traces.shp",
                                 global_offset=None):
    """Export 3D traces with optional coordinate-offset restoration. The LAS-writing section below is retained as commented code. global_offset is [x_offset, y_offset, z_offset]; pass the removed input offset to restore world coordinates, or None to retain the processed coordinates."""
    output_dir = os.path.dirname(las_filename)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ==========================================
    # 0. Restore global coordinates if an offset is supplied.
    # ==========================================
    if global_offset is not None:
        global_offset = np.array(global_offset, dtype=np.float64)
        print(f"Restoring global offset: {global_offset}; restoring point-cloud and trace coordinates...")
        pts_to_save = plane_points + global_offset
    else:
        pts_to_save = plane_points

    # ==========================================
    # 1. Save a LAS point cloud (set header scales and offsets).
    # ==========================================
    # print(f"\n[1/2] Saving point cloud to {las_filename} ...")
    # header = laspy.LasHeader(point_format=2, version="1.2")
    #
    # # Step 1: Set the scale; 0.001 corresponds to millimeters and 0.01 to centimeters for meter coordinates.
    # header.scales = np.array([0.001, 0.001, 0.001])
    #
    # # Step 2: Use coordinate minima as internal LAS offsets.
    # # Actual coordinates = internal integers * scale + offset, avoiding integer overflow.
    # las_internal_offset = np.min(pts_to_save, axis=0)
    # header.offsets = las_internal_offset
    #
    # las = laspy.LasData(header)
    #
    # # Assign XYZ coordinates.
    # las.x = pts_to_save[:, 0]
    # las.y = pts_to_save[:, 1]
    # las.z = pts_to_save[:, 2]
    #
    # # Assign RGB colors.
    # if plane_colors is not None:
    #     if plane_colors.dtype in [np.float32, np.float64] or plane_colors.max() <= 1.0:
    #         colors_16bit = np.clip(plane_colors * 65535, 0, 65535).astype(np.uint16)
    #     else:
    #         colors_16bit = np.clip(plane_colors * 256, 0, 65535).astype(np.uint16)
    #
    #     las.red = colors_16bit[:, 0]
    #     las.green = colors_16bit[:, 1]
    #     las.blue = colors_16bit[:, 2]
    #
    # las.write(las_filename)
    # print(f" -> Point cloud saved. LAS header offset: {header.offsets}")

    # ==========================================
    # 2. Save segments as a 3D Shapefile.
    # ==========================================
    print(f"[2/2] Saving 3D traces to {shp_filename} ...")
    with shapefile.Writer(shp_filename, shapeType=shapefile.POLYLINEZ) as w:
        w.field("ID", "N")
        w.field("Length", "F", decimal=3)

        for i, segment in enumerate(line_segments):
            p_start = np.array(segment[0])
            p_end = np.array(segment[1])

            # Restore the global offset to line endpoints; Shapefile coordinates are 64-bit floats.
            if global_offset is not None:
                p_start += global_offset
                p_end += global_offset

            length = np.linalg.norm(p_start - p_end)

            w.linez([[p_start.tolist(), p_end.tolist()]])
            w.record(ID=i, Length=length)

    print(f" -> 3D Shapefile saved successfully.")

# ================== Export ==================

# Example of an offset removed before processing for Open3D precision:
# original_offset = [523456.0, 4123456.0, 100.0]

# Call the export function; None retains the processed coordinates.
save_las_and_shp_with_offset(
    plane_points=plane_points,
    # plane_colors=colors_plane,
    line_segments=final_lines,
    las_filename=r"outputs/point_cloud.las",
    shp_filename=r"outputs/trace_lines_3d.shp",
    global_offset=None  # To restore world coordinates, pass the removed offset [x, y, z].
)
